# From DICOM to a Generalized Research Dataset
### Companion notebook — *From Pixels to Patients*, AAPM 2026 (Session 1 of 3)

**Brian M. Anderson, PhD** · Radiation Medicine & Applied Sciences

This notebook is the runnable companion to the talk. It takes the public
**NSCLC-Radiomics** collection from **TCIA** and walks the full pipeline the slides
describe, using [**DicomRTTool**](https://github.com/brianmanderson/Dicom_RT_and_Images_to_Mask):

1. **Download** a handful of NSCLC-Radiomics patients (CT + RTSTRUCT) from TCIA.
2. **Discover** every series and the ROIs present.
3. **Survey** the cohort with a metadata *manifest*.
4. **Spot outliers** in spacing and ROI volume from that manifest.
5. **Set a target voxel size** and understand the resampling rules.
6. **Preserve clinical metadata** (age, spacing, acquisition, outcome) as a sidecar.
7. **Convert** everything to **NIfTI** — images, masks, and dose — in one call.
8. **Verify** an exported case and see how to *grow* the dataset over time.

> Everything here runs top-to-bottom. Downloading a few patients takes a few minutes;
> the full collection is ~36 GB, so we default to a small subset.


---
## 0 · Setup

Install the two key packages (plus a few helpers). `DicomRTTool` does the DICOM→NIfTI
work; `tcia_utils` is the official-style client for TCIA's NBIA API.


In [ ]:
# Run once. Restart the kernel afterwards if pip upgrades a loaded package.
%pip install -q --upgrade DicomRTTool tcia_utils SimpleITK pandas matplotlib nibabel

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# One base folder for everything this notebook produces.
BASE      = Path("aapm_nsclc").resolve()
DICOM_DIR = BASE / "dicom"        # raw DICOM downloaded from TCIA
OUT_DIR   = BASE / "nifti"        # generalized NIfTI dataset we will write
MANIFEST  = BASE / "manifest.csv" # cohort survey table
for d in (DICOM_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Working under:", BASE)

---
## 1 · Download NSCLC-Radiomics from TCIA

**Collection:** `NSCLC-Radiomics` (Aerts et al.) — 422 non-small-cell lung cancer
patients, pre-treatment **CT** with radiation-oncologist **RTSTRUCT** delineations
(GTV, lungs, esophagus, heart, spinal cord) and clinical outcomes.

- **DOI:** [10.7937/K9/TCIA.2015.PF0M9REI](https://doi.org/10.7937/K9/TCIA.2015.PF0M9REI)
- **License:** CC BY-NC 3.0 — free for non-commercial research **with attribution**.
  Please cite the collection and the Aerts et al. *Nature Communications* 2014 paper.
- **Full size:** ~35.8 GB. We download a small subset so the notebook is quick to run.

### Method A — programmatic (used here)

`tcia_utils.nbia` queries series metadata and downloads the DICOM for you.
No login is needed for this public collection.


In [ ]:
from tcia_utils import nbia

# Pull *series-level* metadata for the whole collection (fast — metadata only).
series_df = nbia.getSeries(collection="NSCLC-Radiomics", format="df")
print(f"{len(series_df)} series across {series_df['PatientID'].nunique()} patients")
series_df[["PatientID", "Modality", "SeriesInstanceUID"]].head()

Pick a few patients and grab **all** of their series (both the CT and its RTSTRUCT).
Bump `PATIENTS` up, or swap in specific IDs like `"LUNG1-005"`, to pull more.


In [ ]:
N_PATIENTS = 40
PATIENTS = sorted(series_df["PatientID"].unique())[:N_PATIENTS]
print("Downloading:", PATIENTS)

subset = series_df[series_df["PatientID"].isin(PATIENTS)]
print(subset[["PatientID", "Modality"]].value_counts().to_string())

# Downloads into DICOM_DIR/<SeriesInstanceUID>/*.dcm
nbia.downloadSeries(subset, input_type="df", path=str(DICOM_DIR))
print("Done. DICOM under:", DICOM_DIR)

### Method B — NBIA Data Retriever (for the full cohort)

For large pulls, TCIA recommends the desktop **NBIA Data Retriever**:

1. On the [NSCLC-Radiomics page](https://www.cancerimagingarchive.net/collection/nsclc-radiomics/),
   click **Download** to get a `.tcia` *manifest* file.
2. Install the free **NBIA Data Retriever** and open the manifest; it fetches every
   series to a folder you choose.
3. Point `DICOM_DIR` (above) at that folder and skip Method A.

`tcia_utils` can also consume that manifest directly:

```python
nbia.downloadSeries("NSCLC-Radiomics.tcia", input_type="manifest", path=str(DICOM_DIR))
```


---
## 2 · Discover — walk the folders

`DicomReaderWriter.walk_through_folders` recursively scans the tree, groups files by
`SeriesInstanceUID`, and links each RTSTRUCT to its image series — they don't have to
live in the same folder.


In [ ]:
from DicomRTTool.ReaderWriter import DicomReaderWriter, ROIAssociationClass

reader = DicomReaderWriter()
reader.walk_through_folders(str(DICOM_DIR))

# What ROI names exist across everything we downloaded?
rois = reader.return_rois(print_rois=True)

---
## 3 · Survey — write a metadata manifest

`create_manifest` writes **one row per series**: the identifiers, the image spacing
(`spacing_x/y/z`), and **one `<roi> cc` column per ROI** giving its mask volume in cubic
centimetres (blank when that ROI is absent). This is the table the talk builds on.


In [ ]:
reader.create_manifest(str(MANIFEST))

manifest = pd.read_csv(MANIFEST)
print("Shape:", manifest.shape)
manifest.head()

---
## 4 · Spot the outliers

The manifest is your first QC pass. Three things to look for (slide: *Spotting the outliers*):

- **Geometry outliers** — an odd `spacing_z` (e.g. a 5 mm slice in a 3 mm cohort).
- **Volume outliers** — an ROI far from the cohort norm (a partial or mis-drawn contour).
- **Missing structures** — a blank `cc` cell means that ROI isn't on the series.

Sort the columns, or plot the distributions — outliers sit in the tails.


In [ ]:
# Spacing overview
spacing_cols = [c for c in manifest.columns if c.startswith("spacing_")]
print(manifest[spacing_cols].describe().round(3).to_string())

# ROI volume columns are everything ending in ' cc'
cc_cols = [c for c in manifest.columns if c.strip().endswith("cc")]
print("\nROI volume columns:", cc_cols)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

# (a) slice-thickness distribution
axes[0].hist(manifest["spacing_z"].dropna(), bins=20, color="#007DBA")
axes[0].set_title("spacing_z (mm)")
axes[0].set_xlabel("slice thickness"); axes[0].set_ylabel("series")

# (b) volume distribution for the first ROI that has data
vol_col = next((c for c in cc_cols if manifest[c].notna().sum() > 1), None)
if vol_col:
    axes[1].hist(manifest[vol_col].dropna(), bins=20, color="#1C7293")
    axes[1].set_title(f"{vol_col}")
    axes[1].set_xlabel("volume (cc)"); axes[1].set_ylabel("series")
plt.tight_layout(); plt.show()

In [ ]:
def flag_outliers(series, k=1.5):
    # Boolean mask of IQR outliers: values beyond k*IQR below Q1 or above Q3.
    s = series.dropna()
    if len(s) < 4:
        return pd.Series(False, index=series.index)
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return (series < lo) | (series > hi)

# Flag geometry + volume outliers so you can review or exclude them before converting.
report = manifest[["patient_hash", "series_hash", "spacing_z"] + cc_cols].copy()
report["spacing_z_outlier"] = flag_outliers(manifest["spacing_z"])
for c in cc_cols:
    report[c + "_outlier"] = flag_outliers(manifest[c])

flagged = report[report.filter(like="_outlier").any(axis=1)]
print(f"{len(flagged)} series flagged for review")
flagged.head(20)

---
## 5 · Select ROIs & normalize their names

Real ROI names are inconsistent across a cohort (`Lung-Left`, `lung_l`, `left lung`).
`ROIAssociationClass` maps aliases onto one canonical name;
`set_contour_names_and_associations` picks the ROIs you actually want. Adjust the aliases
to whatever `return_rois()` printed above.


In [ ]:
reader.set_contour_names_and_associations(
    contour_names=["gtv", "lung_l", "lung_r", "esophagus", "heart", "cord"],
    associations=[
        ROIAssociationClass("gtv",       ["gtv-1", "gtv1", "gtv"]),
        ROIAssociationClass("lung_l",    ["lung-left", "lung_l", "left lung"]),
        ROIAssociationClass("lung_r",    ["lung-right", "lung_r", "right lung"]),
        ROIAssociationClass("esophagus", ["esophagus"]),
        ROIAssociationClass("heart",     ["heart"]),
        ROIAssociationClass("cord",      ["spinal-cord", "spinalcord", "spinal cord"]),
    ],
)

# Which series carry the selected ROIs?
print("Series with the selected ROIs:", reader.indexes_with_contours)

---
## 6 · Set the output voxel size

The survey showed native spacing varies. Pick **one** target so every case lands on the
same grid (slide: *Choosing an output voxel size*).

- **Isotropic** `(1.0, 1.0, 1.0)` — uniform, model-friendly, larger arrays.
- **Preserve native-ish** `(0.98, 0.98, 3.0)` — smaller, but mixed geometry.

`write_to_folder` applies the rule automatically: **Linear** for image and dose,
**Nearest-neighbour** for masks (never blur a label). Dose lands on the resampled image grid.


In [ ]:
OUTPUT_SPACING = (1.0, 1.0, 3.0)   # (x, y, z) in mm — set to None to keep native spacing

# The public resampling helpers, if you ever need them directly:
# from DicomRTTool import resample_to_spacing, resample_to_reference
print("Target voxel size (mm):", OUTPUT_SPACING)

---
## 7 · Preserve clinical metadata

Pixels alone aren't research-ready. Ask the reader to pull extra DICOM tags by name;
`write_to_folder` records them in a **`metadata.json`** beside every exported series
(slide: *Preserving clinical metadata*).

As of the schema-v2 update, that sidecar is a **grouped, versioned document**
(`"schema_version": 2`). The DICOM features the walk already parsed are organized by
category — `image`, `structures`, `doses`, `plans` — and **your requested tags land
inside the owning category's `tags` sub-dict** (image tags under `image["tags"]`).
Categories with no matching files are omitted, so an image-only series still parses
cleanly with `meta.get("structures", [])`, and identical tag names across categories no
longer overwrite each other.

SITK image tags use `"group|element"` strings. Pull only non-identifying tags — values are
written verbatim, so never pull `PatientName`/`PatientID` into an anonymized export.
Pass `metadata_style="flat"` to `write_to_folder` for the historical `{name: value}` dict.

In [ ]:
EXTRA_TAGS = {
    "PatientAge":            "0010|1010",
    "PatientSex":            "0010|0040",
    "Manufacturer":          "0008|0070",
    "ManufacturerModelName": "0008|1090",
    "KVP":                   "0018|0060",
    "SliceThickness":        "0018|0050",
}

# Rebuild the reader with the same ROI selection *plus* the extra tags and anonymization.
reader = DicomReaderWriter(
    Contour_Names=["gtv", "lung_l", "lung_r", "esophagus", "heart", "cord"],
    associations=[
        ROIAssociationClass("gtv",       ["gtv-1", "gtv1", "gtv"]),
        ROIAssociationClass("lung_l",    ["lung-left", "lung_l", "left lung"]),
        ROIAssociationClass("lung_r",    ["lung-right", "lung_r", "right lung"]),
        ROIAssociationClass("esophagus", ["esophagus"]),
        ROIAssociationClass("heart",     ["heart"]),
        ROIAssociationClass("cord",      ["spinal-cord", "spinalcord", "spinal cord"]),
    ],
    image_sitk_string_keys=EXTRA_TAGS,
    require_all_contours=False,   # keep series that carry only some of the ROIs
)
reader.walk_through_folders(str(DICOM_DIR))
print("Reader ready. Series with contours:", reader.indexes_with_contours)

---
## 8 · Convert to NIfTI — one call

`write_to_folder` exports every selected series to a tidy per-case tree: resampled
image, one mask per ROI, dose (if present), and the grouped **`metadata.json`** sidecar —
plus a cohort `manifest.csv` and, because we anonymize, an `anonymization_key.json`
(slide: *DICOM → NIfTI in one call*).

In [ ]:
reader.write_to_folder(
    str(OUT_DIR),
    output_spacing=OUTPUT_SPACING,       # resample on the way out
    anonymize=True,
    salt="AAPM2026-NSCLC",               # deterministic SHA-256 hashing
    metadata_style="grouped",            # schema v2 (default); "flat" = old {name: value} dict
)
print("Exported to:", OUT_DIR)

In [ ]:
# Show the resulting tree (first couple of levels).
def show_tree(root: Path, max_files=6, prefix=""):
    entries = sorted(root.iterdir())
    dirs  = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    for f in files[:max_files]:
        print(prefix + f.name)
    for d in dirs:
        print(prefix + d.name + "/")
        show_tree(d, max_files, prefix + "    ")

show_tree(OUT_DIR)

---
## 9 · Verify an exported case

Load one exported image and a mask, confirm they share geometry (same size / spacing
after resampling), and read the grouped `metadata.json` — the requested tags live under
`image["tags"]`, and each exported ROI carries its `volume_cc` and `exported_file`.

In [ ]:
import SimpleITK as sitk, json as _json

# Find the first exported case that has an image + at least one mask.
cases = [p.parent for p in OUT_DIR.rglob("image.nii.gz")]
assert cases, "No exported images found — re-run section 8."
case = cases[0]
print("Inspecting:", case.relative_to(OUT_DIR))

img = sitk.ReadImage(str(case / "image.nii.gz"))
print("Image size   :", img.GetSize())
print("Image spacing:", tuple(round(s, 3) for s in img.GetSpacing()))

mask_files = sorted((case / "masks").glob("*.nii.gz")) if (case / "masks").exists() else []
print("Masks        :", [m.name.replace(".nii.gz", "") for m in mask_files])

# --- Read the grouped metadata.json (schema v2) ---
meta = _json.loads((case / "metadata.json").read_text())
print("\nschema_version:", meta["schema_version"], "| anonymized:", meta["anonymized"])

image = meta["image"]                       # the image block is always present
print("image.export :", image.get("export"))     # effective spacing + resampled flag
print("image.tags   :", image.get("tags", {}))   # <- your requested DICOM tags live here

# Categories with no matching files are omitted -> use .get() with a default.
for rt in meta.get("structures", []):
    for roi in rt["rois"]:
        if "exported_file" in roi:          # only exported ROIs carry volume + file
            name = roi.get("canonical_name", roi["name"])
            print(f"  ROI {roi['name']:<12} -> {name:<10} {roi['volume_cc']:8.1f} cc"
                  f"  ->  {roi['exported_file']}")
print("doses:", len(meta.get("doses", [])), "| plans:", len(meta.get("plans", [])))

In [ ]:
# Overlay the first mask on the image at the slice where the mask is largest.
if mask_files:
    arr  = sitk.GetArrayFromImage(img)                       # (z, y, x)
    mask = sitk.GetArrayFromImage(sitk.ReadImage(str(mask_files[0])))
    z = int(np.argmax(mask.sum(axis=(1, 2)))) if mask.any() else arr.shape[0] // 2

    plt.figure(figsize=(5, 5))
    plt.imshow(arr[z], cmap="gray")
    plt.imshow(np.ma.masked_where(mask[z] == 0, mask[z]), cmap="autumn", alpha=0.45)
    plt.title(f"{case.name} — {mask_files[0].stem.replace('.nii','')} @ z={z}")
    plt.axis("off"); plt.show()
else:
    print("No masks exported for this case.")

---
## 10 · Grow the dataset over time

The `anonymization_key.json` maps each study hash back to its MRN — **stored securely,
never traveling with the data**. Because the hashing is deterministic (same `salt` →
same hash), you can re-pull the same patients later and fold new imaging or follow-up in
without re-identifying anything.

`create_manifest` is **incremental**: point it at an existing manifest and it *upserts*
by `series_hash` — existing rows are updated, new series appended — so one manifest can
grow across many walks.

```python
# Later, after downloading more patients into DICOM_DIR:
reader.reset()
reader.walk_through_folders(str(DICOM_DIR))
reader.create_manifest(str(MANIFEST))   # updates the same file in place
```

### Where this goes next
- **Session 2** builds reproducible PyTorch pipelines on this exact NIfTI dataset.
- **Session 3** covers integrating and monitoring models in the clinic.

**Tool:** <https://github.com/brianmanderson/Dicom_RT_and_Images_to_Mask> · `pip install DicomRTTool`
